# Graph coloring on the Washington map

This notebook uses `color.py` to turn Washington's counties (and congressional districts) into an
adjacency graph, then shows how each coloring algorithm works on it:

1. Load the map and build the dual graph
2. Compare the heuristic colorings with the exact minimum coloring
3. Animate greedy / DSatur step by step
4. Animate the exact backtracking search
5. Show why the lower bound holds, and how much vertex order matters
6. Show how the boundary source (TIGER vs. cartographic) changes the graph
7. Congressional districts, plus which counties are split between districts

**Requirements:** `pip install geopandas networkx matplotlib requests`. The first run downloads Census
shapefiles into `./data/` (the national TIGER county file is about 80 MB). Later runs read from that cache.

In [ ]:
import itertools, random, time
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from matplotlib import animation
from matplotlib.patches import Patch
from IPython.display import HTML, display

import color as wc  # the starter module (color.py, same folder)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["animation.embed_limit"] = 60  # MB, for the inline animations
UNCOLORED = "#e6e6e6"

## 1. Load the counties and build the graph

`SOURCE` selects the boundary file:

* `"tiger"`: legal boundaries that extend into water, so island counties (San Juan, Island) touch their
  neighbours across Puget Sound.
* `"cb"`: cartographic boundaries clipped to the shoreline. San Juan becomes an isolated node.

`MODE = "rook"` is the standard map-coloring rule: two regions conflict only if they share a border
segment, not just a corner point.

In [ ]:
LAYER, SOURCE, MODE = "county", "tiger", "rook"
MIN_SHARED_M, SNAP_M = 100.0, 0.0

cfg = wc.SOURCES[LAYER]
ID, NAME = cfg["id_col"], cfg["name_col"]
gdf = wc.fetch_layer(LAYER, SOURCE)
G = wc.build_graph(gdf, ID, NAME, MODE, MIN_SHARED_M, SNAP_M)
names = nx.get_node_attributes(G, "name")
pos = {n: (d["x"], d["y"]) for n, d in G.nodes(data=True)}

print(f"{G.number_of_nodes()} regions, {G.number_of_edges()} shared borders")
print("planar:", nx.check_planarity(G)[0], "| max degree:", max(d for _, d in G.degree))
print("isolated:", [names[n] for n in nx.isolates(G)] or "none")

In [ ]:
def paint(coloring, ax, title="", highlight=(), edge_hl=(), labels=False):
    """Fill each region with its color class; regions missing from `coloring` are gray."""
    fills = gdf[ID].map(
        lambda i: wc.PALETTE[coloring[i] % len(wc.PALETTE)] if i in coloring else UNCOLORED
    )
    gdf.plot(color=fills, edgecolor="white", linewidth=0.8, ax=ax)
    if highlight:
        gdf[gdf[ID].isin(highlight)].boundary.plot(ax=ax, color="black", linewidth=2.2)
    if edge_hl:
        nx.draw_networkx_edges(G, pos, edgelist=list(edge_hl), ax=ax, width=2.5, edge_color="black")
    if labels:
        for n, (x, y) in pos.items():
            ax.annotate(names[n], (x, y), ha="center", va="center", fontsize=5.5)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()


def legend(k, ax, loc="lower left"):
    ax.legend(handles=[Patch(color=wc.PALETTE[c], label=f"color {c}") for c in range(k)],
              loc=loc, fontsize=7, frameon=False)

### The dual graph: one node per county, one edge per shared border

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7.5))
gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", linewidth=0.6, ax=ax)
nx.draw_networkx_edges(G, pos, ax=ax, width=1.1, edge_color="#444444")
deg = dict(G.degree)
nodes = nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[60 + 25 * deg[n] for n in G],
                               node_color=[deg[n] for n in G], cmap="viridis")
nx.draw_networkx_labels(G, pos, labels=names, ax=ax, font_size=6)
fig.colorbar(nodes, ax=ax, shrink=0.6, label="degree (number of neighbours)")
ax.set_title(f"WA {LAYER} adjacency graph ({SOURCE}, {MODE})")
ax.set_axis_off()
plt.show()

## 2. Every algorithm side by side

In [ ]:
results, timings = {}, {}
for algo, strategy in wc.ALGORITHMS.items():
    t0 = time.perf_counter()
    col = nx.greedy_color(G, strategy=strategy)
    timings[algo] = time.perf_counter() - t0
    results[algo] = {"coloring": col, "k": len(set(col.values()))}

t0 = time.perf_counter()
col, k = wc.exact_coloring(G)
timings["exact"] = time.perf_counter() - t0
results["exact"] = {"coloring": col, "k": k}

for a, r in results.items():
    assert not wc.is_proper(G, r["coloring"]), f"{a} produced a conflict"

pd.DataFrame({
    "colors used": {a: r["k"] for a, r in results.items()},
    "time (ms)": {a: round(1000 * t, 2) for a, t in timings.items()},
    "class sizes": {a: sorted(Counter(r["coloring"].values()).values(), reverse=True)
                    for a, r in results.items()},
})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (a, r) in zip(axes.flat, results.items()):
    paint(r["coloring"], ax, f"{a}: {r['k']} colors")
    legend(r["k"], ax)
axes.flat[-1].set_axis_off()
fig.suptitle(f"WA {LAYER}: colorings by algorithm", fontsize=13)
plt.tight_layout()
plt.show()

Colorings can use the same number of colors and still be different maps. Here we check which algorithms
produce the **same partition** (identical color classes once you ignore which color is which).

In [ ]:
parts = {a: wc.canonical_partition(r["coloring"]) for a, r in results.items()}
algos = list(parts)
pd.DataFrame([[parts[a] == parts[b] for b in algos] for a in algos], index=algos, columns=algos)

## 3. Step-by-step greedy coloring

Greedy coloring visits the vertices in some order and gives each one the **smallest color not already
used by a colored neighbour**. The strategy only changes the order:

* `largest_first`: highest degree first (a fixed order)
* `dsatur`: next vertex is the one with the most *distinct* neighbour colors (recomputed every step)
* `smallest_last`: repeatedly remove the min-degree vertex, then color in reverse removal order

`nx.greedy_color` fills its result dict in visit order, which lets us replay it.

In [ ]:
def greedy_frames(algo):
    col = results[algo]["coloring"]
    order = list(col)  # insertion order == visit order
    return [(dict((v, col[v]) for v in order[: i + 1]), order[i]) for i in range(len(order))]


def animate(frames, title_fn, interval=350):
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    def draw(i):
        ax.clear()
        coloring, current = frames[i][:2]
        paint(coloring, ax, title_fn(i), highlight=[current] if current else ())
    anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(anim.to_jshtml())


ALGO = "dsatur"  # try "largest_first", "smallest_last", "independent_set"
frames = greedy_frames(ALGO)
animate(frames, lambda i: f"{ALGO} step {i + 1}/{len(frames)}: "
                          f"{names[frames[i][1]]} gets color {frames[i][0][frames[i][1]]}")

A static version for reports: the visit order is printed on each county, so you can see where the
algorithm starts and how it spreads.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, algo in zip(axes, ["largest_first", "dsatur", "smallest_last"]):
    col = results[algo]["coloring"]
    paint(col, ax, f"{algo} ({results[algo]['k']} colors), numbers = visit order")
    for step, v in enumerate(col, 1):
        ax.annotate(str(step), pos[v], ha="center", va="center", fontsize=6.5, weight="bold")
plt.tight_layout()
plt.show()

## 4. The exact search (backtracking)

`wc.exact_coloring` tries k = (max clique size), k+1, … and for each k runs a DSatur-ordered
backtracking search. The function below is the same search as `wc._try_k` with **tracing added**, so we
can watch it assign colors and undo them.

In [ ]:
def traced_try_k(G, k, max_events=5000):
    colors, events = {}, []

    def pick():
        best, key = None, None
        for v in G:
            if v in colors:
                continue
            sat = len({colors[u] for u in G[v] if u in colors})
            cand = (sat, G.degree(v))
            if key is None or cand > key:
                best, key = v, cand
        return best

    def rec():
        v = pick()
        if v is None:
            return True
        used = {colors[u] for u in G[v] if u in colors}
        max_open = max(colors.values(), default=-1) + 1
        for c in range(min(k, max_open + 1)):
            if c not in used:
                colors[v] = c
                if len(events) < max_events:
                    events.append((dict(colors), v, "assign"))
                if rec():
                    return True
                del colors[v]
                if len(events) < max_events:
                    events.append((dict(colors), v, "undo"))
        return False

    ok = rec()
    return (dict(colors) if ok else None), events


lower = max(len(c) for c in nx.find_cliques(G))
traces = {}
for kk in range(lower, results["exact"]["k"] + 1):
    sol, ev = traced_try_k(G, kk)
    traces[kk] = ev
    undos = sum(e[2] == "undo" for e in ev)
    print(f"k={kk}: {'solved' if sol else 'impossible'} after {len(ev)} steps ({undos} undos)")

In [ ]:
K_SHOW = results["exact"]["k"]  # or `lower` to watch a failing search (it may be long)
ev = traces[K_SHOW][:400]
animate(ev, lambda i: f"backtracking, k={K_SHOW}: step {i + 1}/{len(ev)} {ev[i][2]} {names[ev[i][1]]}",
        interval=200)

## 5. Why can't we use fewer colors?

The **maximum clique** gives a lower bound: regions that all touch each other need distinct colors.
If the exact search needs more colors than the clique size, the obstruction is something else, usually
an **odd wheel** (a hub surrounded by an odd cycle of neighbours). The cycle needs 3 colors and the hub
needs a 4th.

In [ ]:
clique = max(nx.find_cliques(G), key=len)
chi = results["exact"]["k"]
print(f"max clique ({len(clique)}):", [names[n] for n in clique], f"| chromatic number: {chi}")

def odd_wheels(G):
    """Hubs whose neighbourhood contains an odd cycle (so hub + cycle needs 4 colors)."""
    for hub in G:
        H = G.subgraph(G[hub])
        if not nx.is_bipartite(H):
            cyc = next(c for c in nx.simple_cycles(H) if len(c) % 2 == 1)
            yield hub, cyc

wheel = next(odd_wheels(G), None) if chi > len(clique) else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
paint(results["exact"]["coloring"], axes[0], f"max clique of size {len(clique)} (outlined)",
      highlight=clique)
if wheel:
    hub, cyc = wheel
    paint(results["exact"]["coloring"], axes[1],
          f"odd wheel: hub {names[hub]} + {len(cyc)}-cycle", highlight=[hub, *cyc],
          edge_hl=[(hub, c) for c in cyc] + list(zip(cyc, cyc[1:] + cyc[:1])))
else:
    axes[1].set_axis_off()
    axes[1].set_title("clique bound equals chromatic number: no odd wheel needed")
plt.tight_layout()
plt.show()

### Vertex order matters

Plain greedy with a **random** vertex order. Every run is a valid coloring, but the number of colors
depends on the order. The ordering heuristics above exist to steer away from the bad orders.

In [ ]:
rng = random.Random(0)
counts = Counter()
worst = None
for _ in range(2000):
    order = list(G)
    rng.shuffle(order)
    col = nx.greedy_color(G, strategy=lambda _G, _c, o=order: iter(o))
    k = len(set(col.values()))
    counts[k] += 1
    if worst is None or k > worst[0]:
        worst = (k, col)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.6]})
ks = sorted(counts)
a1.bar([str(k) for k in ks], [counts[k] for k in ks], color="#4C78A8")
a1.set_xlabel("colors used"); a1.set_ylabel("runs (of 2000)")
a1.set_title("greedy with random vertex orders")
paint(worst[1], a2, f"worst random order found: {worst[0]} colors")
legend(worst[0], a2)
plt.tight_layout()
plt.show()

## 6. How the boundary data changes the graph

The same state gives different graphs depending on the boundary file and the adjacency rule. Black edges
are present in one version and missing from the other.

In [ ]:
def graph_for(source=SOURCE, mode=MODE, min_shared_m=MIN_SHARED_M):
    g = wc.fetch_layer(LAYER, source)
    return wc.build_graph(g, ID, NAME, mode, min_shared_m, SNAP_M)

def edge_set(g):
    return {frozenset(e) for e in g.edges}

variants = {
    "tiger / rook": graph_for("tiger", "rook"),
    "cb / rook": graph_for("cb", "rook"),
    "tiger / queen": graph_for("tiger", "queen"),
}
base = edge_set(variants["tiger / rook"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for ax, name in zip(axes, ["cb / rook", "tiger / queen"]):
    other = edge_set(variants[name])
    only_base, only_other = base - other, other - base
    gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", linewidth=0.5, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.8, edge_color="#bbbbbb")
    nx.draw_networkx_edges(G, pos, edgelist=[tuple(e) for e in only_base], ax=ax,
                           width=2.5, edge_color="#E45756", label="only in tiger / rook")
    nx.draw_networkx_edges(G, pos, edgelist=[tuple(e) for e in only_other], ax=ax,
                           width=2.5, edge_color="#4C78A8", style="dashed", label=f"only in {name}")
    ax.set_title(f"tiger / rook vs {name}: -{len(only_base)} / +{len(only_other)} edges")
    ax.legend(loc="lower left", fontsize=8, frameon=False)
    ax.set_axis_off()
    for e in only_base | only_other:
        print(f"{name:>14}: {'missing' if e in only_base else 'extra  '} "
              + " - ".join(names[n] for n in e))
plt.tight_layout()
plt.show()

pd.DataFrame({name: {"edges": g.number_of_edges(),
                     "isolated": len(list(nx.isolates(g))),
                     "chromatic number": wc.exact_coloring(g)[1]}
              for name, g in variants.items()}).T

## 7. Congressional districts

In [ ]:
cd_cfg = wc.SOURCES["cd"]
cd_gdf = wc.fetch_layer("cd", SOURCE)
G_cd = wc.build_graph(cd_gdf, cd_cfg["id_col"], cd_cfg["name_col"], MODE, MIN_SHARED_M, SNAP_M)
cd_col, cd_k = wc.exact_coloring(G_cd)
cd_pos = {n: (d["x"], d["y"]) for n, d in G_cd.nodes(data=True)}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.5))
cd_gdf.plot(color=cd_gdf[cd_cfg["id_col"]].map(lambda i: wc.PALETTE[cd_col[i]]),
            edgecolor="white", linewidth=1, ax=a1)
for n, (x, y) in cd_pos.items():
    a1.annotate(G_cd.nodes[n]["name"], (x, y), ha="center", fontsize=8, weight="bold")
a1.set_title(f"WA congressional districts: exact coloring, {cd_k} colors"); a1.set_axis_off()

cd_gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", ax=a2)
nx.draw_networkx(G_cd, cd_pos, ax=a2, node_color=[wc.PALETTE[cd_col[n]] for n in G_cd],
                 labels=nx.get_node_attributes(G_cd, "name"), font_size=7, node_size=500)
a2.set_title(f"district graph: {G_cd.number_of_nodes()} nodes, {G_cd.number_of_edges()} edges")
a2.set_axis_off()
plt.tight_layout()
plt.show()

### Counties split across districts

`wc.county_district_overlay` intersects the two layers (ignoring slivers under 1 km²) and lists every
county that falls in more than one district.

In [ ]:
split = wc.county_district_overlay(SOURCE)
display(pd.DataFrame({"districts": {c: ", ".join(d) for c, d in split.items()}}))

fig, ax = plt.subplots(figsize=(10, 6.5))
gdf.plot(color=gdf[NAME].map(lambda n: "#F58518" if n in split else "#f2f2f2"),
         edgecolor="#9a9a9a", linewidth=0.5, ax=ax)
cd_gdf.boundary.plot(ax=ax, color="black", linewidth=1.4)
ax.set_title(f"{len(split)} counties (orange) are split by district lines (black)")
ax.set_axis_off()
plt.show()

## 8. (Optional) Analyze a published map's coloring

If you transcribe the colors of a real WA county map into `{county name: color}`, `wc.analyze_reference`
checks that it is proper, compares it with the chromatic number, and tests whether plain greedy
could have produced it. The example below just recolors our DSatur result with color names; replace it
with your transcription.

In [ ]:
ref = {names[n]: ["blue", "orange", "green", "red", "purple", "brown", "teal"][c]
       for n, c in results["dsatur"]["coloring"].items()}
# ref = json.load(open("ref_colors.json"))
wc.analyze_reference(G, ref, results)